# 09 - Computer Networks security semantics and recovery audit

Produces the binary, attack-family, exact-type, attack-to-benign, benign-to-attack, response-equivalence proxy, and post-head-refit calibration tables requested by the journal-facing audit.

**Safety:** this notebook writes only new files under `results/tables/comnet/` and does not overwrite archived manuscript result tables.

In [ ]:
# Colab/bootstrap cell: no tokens or credentials are required.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

import os, sys, json, time
from pathlib import Path

REPO = Path('/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression')
if not REPO.exists():
    # Local/Jupyter fallback: run the notebook from the repository root.
    REPO = Path.cwd()
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from src.config import CFG, PATHS, set_all_seeds
set_all_seeds(CFG['anchor_seed'])

OUT_TABLE = PATHS.tables('comnet')
OUT_TABLE.mkdir(parents=True, exist_ok=True)
print('Repository:', REPO)
print('Outputs:', OUT_TABLE)


## Configuration
The default path evaluates existing checkpoints. Set `REBUILD_MISSING=True` only when a compressed checkpoint is absent.

In [ ]:
DATASET = 'ciciot2023'
ARCH = 'cnn1d'
ARCH_KW = {'channels': (64, 128)}
SEED = int(CFG['anchor_seed'])
REBUILD_MISSING = False
CELLS = ['M0', 'prune50', 'prune80', 'distillation', 'int8', 'float16']


## Load the provenance-ordered dataset and baseline

In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import f1_score

from src.data import load_raw, clean, temporal_within_capture_split
from src.train import load_anchor, feature_columns
from src import compression as comp
from src import mitigate
from src.comnet_audit import (security_semantics, calibration_summary, family_mapping_table,
                              environment_record, write_json)

df = clean(load_raw(DATASET, subsample=True, seed=SEED), DATASET)
splits = temporal_within_capture_split(df, SEED)
m0, le, scaler, feat_cols = load_anchor(DATASET, ARCH, 'M0', SEED, arch_kwargs=ARCH_KW)
family_map_table = family_mapping_table(le.classes_)
family_map_table.to_csv(OUT_TABLE / 'ciciot2023_alert_family_mapping.csv', index=False)
display(family_map_table)
print(len(df), 'rows |', len(le.classes_), 'classes |', len(feat_cols), 'features')


## Load or derive compression cells

In [ ]:
def load_or_build(cell):
    if cell == 'M0':
        return {'model': m0, 'note': 'loaded M0', 'is_half': False, 'is_int8': False}
    if cell == 'int8':
        model, note = comp.to_int8(m0, ARCH)
        return {'model': model, 'note': note, 'is_half': False, 'is_int8': True}
    if cell == 'float16':
        return {'model': comp.to_float16(m0), 'note': 'derived fp16', 'is_half': True, 'is_int8': False}
    kwargs = {'channels': (24, 48)} if cell == 'distillation' else ARCH_KW
    try:
        model, _le, _scaler, _feat = load_anchor(DATASET, ARCH, cell, SEED, arch_kwargs=kwargs)
        if list(_le.classes_) != list(le.classes_) or list(_feat) != list(feat_cols):
            raise RuntimeError(f'{cell}: checkpoint schema differs from M0')
        return {'model': model, 'note': f'loaded {cell}', 'is_half': False, 'is_int8': False}
    except Exception as exc:
        if not REBUILD_MISSING:
            print(f'SKIP {cell}: {exc}')
            return None
        if cell.startswith('prune'):
            amount = int(cell.replace('prune','')) / 100
            model, _, _ = comp.prune_and_finetune(m0, df, DATASET, splits, SEED, amount, arch=ARCH)
        elif cell == 'distillation':
            model, _, _ = comp.distill(df, DATASET, splits, SEED, m0,
                                       student_kwargs={'channels': (24, 48)}, arch=ARCH)
        else:
            raise
        return {'model': model, 'note': f'rebuilt {cell}', 'is_half': False, 'is_int8': False}

matrix = {c: load_or_build(c) for c in CELLS}
matrix = {k: v for k, v in matrix.items() if v is not None}
{k: v['note'] for k, v in matrix.items()}


## Binary, family, fine-type, and alert-semantic metrics

In [ ]:
summary_rows, family_rows, substitution_rows, calibration_rows = [], [], [], []

pred_cache = {}
for cell, entry in matrix.items():
    rec, macro, probs, y_true, y_pred = comp.evaluate_cell(
        entry, df, splits, le, scaler, feat_cols, which='test'
    )
    pred_cache[cell] = {'probs': probs, 'y_true': y_true, 'y_pred': y_pred}
    s, f, sub = security_semantics(y_true, y_pred, le.classes_)
    s.insert(0, 'cell', cell); summary_rows.append(s)
    f.insert(0, 'cell', cell); family_rows.append(f)
    sub.insert(0, 'cell', cell); substitution_rows.append(sub)
    c = calibration_summary(probs, y_true); c.insert(0, 'cell', cell); calibration_rows.append(c)

security_summary = pd.concat(summary_rows, ignore_index=True)
family_metrics = pd.concat(family_rows, ignore_index=True)
substitution_metrics = pd.concat(substitution_rows, ignore_index=True)
calibration_metrics = pd.concat(calibration_rows, ignore_index=True)

display(security_summary.round(4))
display(family_metrics.round(4))
display(substitution_metrics.round(4))
display(calibration_metrics.round(4))

security_summary.to_csv(OUT_TABLE / 'security_semantics_by_cell.csv', index=False)
family_metrics.to_csv(OUT_TABLE / 'family_metrics_by_cell.csv', index=False)
substitution_metrics.to_csv(OUT_TABLE / 'substitution_semantics_by_cell.csv', index=False)
calibration_metrics.to_csv(OUT_TABLE / 'calibration_extended_by_cell.csv', index=False)


## Audit the dense-head repair across the same security and calibration metrics

In [ ]:
if 'prune80' not in matrix:
    raise RuntimeError('A prune80 checkpoint is required for the recovery audit.')

p80 = matrix['prune80']['model']
Lval, yval, Ltest, ytest = mitigate.refit_head(
    p80, df, splits, scaler, feat_cols, le,
    epochs=15, lr=1e-2, batch_size=4096, seed=SEED,
)
probs_refit = torch.softmax(torch.tensor(Ltest), dim=1).numpy()
pred_refit = probs_refit.argmax(1)

s_refit, f_refit, sub_refit = security_semantics(ytest, pred_refit, le.classes_)
s_refit.insert(0, 'cell', 'prune80_head_refit')
f_refit.insert(0, 'cell', 'prune80_head_refit')
sub_refit.insert(0, 'cell', 'prune80_head_refit')
c_refit = calibration_summary(probs_refit, ytest)
c_refit.insert(0, 'cell', 'prune80_head_refit')

display(s_refit.round(4)); display(f_refit.round(4)); display(c_refit.round(4))
s_refit.to_csv(OUT_TABLE / 'security_semantics_head_refit.csv', index=False)
f_refit.to_csv(OUT_TABLE / 'family_metrics_head_refit.csv', index=False)
sub_refit.to_csv(OUT_TABLE / 'substitution_semantics_head_refit.csv', index=False)
c_refit.to_csv(OUT_TABLE / 'calibration_head_refit.csv', index=False)


## Completion checks

In [ ]:
assert set(['attack_to_benign_rate','benign_to_attack_rate','family_accuracy']).issubset(security_summary.columns)
assert (security_summary['binary_attack_recall'].between(0,1)).all()
write_json(OUT_TABLE / 'security_audit_environment.json', environment_record())
print('Saved Computer Networks security-semantic and recovery-audit tables.')
print('The family mapping is a transparent response-equivalence proxy; edit it only with a documented operational rationale.')
